# Analysis of Used Car Listings on eBay Kleinanzeigen
In this notebook, we will be exploring a dataset of used car listings scraped from the German version of eBay. After cleaning the data, we will analyze it to find trends in the data that might give us some insight into the factors that influence car sales.

In [80]:
# https://www.dataquest.io/projects/guided-project-a-exploring-ebay-car-sales-data/
import os
import kagglehub
from dotenv import load_dotenv

# Load the environment variables from .env
load_dotenv()

# Download latest version to datasets directory
kagglehub.dataset_download("sijovm/used-cars-data-from-ebay-kleinanzeigen", output_dir='./datasets')

'./datasets'

In [81]:
# Determine encoding to read file with pandas
import chardet
with open('./datasets/autos.csv', 'rb') as rawdata:
    result = chardet.detect(rawdata.read(100000))
result

{'encoding': 'Windows-1252', 'confidence': 0.73, 'language': ''}

In [82]:
import pandas as pd
import numpy as np

df = pd.read_csv('./datasets/autos.csv', encoding='Windows-1252')

# Explore the data
df.head()

,dateCrawled,name,seller,offerType,price,abtest,vehicleType,yearOfRegistration,gearbox,powerPS,model,kilometer,monthOfRegistration,fuelType,brand,notRepairedDamage,dateCreated,nrOfPictures,postalCode,lastSeen
0,2016-03-24 11:52:17,Golf_3_1.6,privat,Angebot,480,test,NaN,1993,manuell,0,golf,150000,0,benzin,volkswagen,NaN,2016-03-24 00:00:00,0,70435,2016-04-07 03:16:57
1,2016-03-24 10:58:45,A5_Sportback_2.7_Tdi,privat,Angebot,18300,test,coupe,2011,manuell,190,NaN,125000,5,diesel,audi,ja,2016-03-24 00:00:00,0,66954,2016-04-07 01:46:50
2,2016-03-14 12:52:21,"Jeep_Grand_Cherokee_""Overland""",privat,Angebot,9800,test,suv,2004,automatik,163,grand,125000,8,diesel,jeep,NaN,2016-03-14 00:00:00,0,90480,2016-04-05 12:47:46
3,2016-03-17 16:54:04,GOLF_4_1_4__3TÜRER,privat,Angebot,1500,test,kleinwagen,2001,manuell,75,golf,150000,6,benzin,volkswagen,nein,2016-03-17 00:00:00,0,91074,2016-03-17 17:40:17
4,2016-03-31 17:25:20,Skoda_Fabia_1.4_TDI_PD_Classic,privat,Angebot,3600,test,kleinwagen,2008,manuell,69,fabia,90000,7,diesel,skoda,nein,2016-03-31 00:00:00,0,60437,2016-04-06 10:17:21


In [83]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 371528 entries, 0 to 371527
Data columns (total 20 columns):
 #   Column               Non-Null Count   Dtype
---  ------               --------------   -----
 0   dateCrawled          371528 non-null  str  
 1   name                 371528 non-null  str  
 2   seller               371528 non-null  str  
 3   offerType            371528 non-null  str  
 4   price                371528 non-null  int64
 5   abtest               371528 non-null  str  
 6   vehicleType          333659 non-null  str  
 7   yearOfRegistration   371528 non-null  int64
 8   gearbox              351319 non-null  str  
 9   powerPS              371528 non-null  int64
 10  model                351044 non-null  str  
 11  kilometer            371528 non-null  int64
 12  monthOfRegistration  371528 non-null  int64
 13  fuelType             338142 non-null  str  
 14  brand                371528 non-null  str  
 15  notRepairedDamage    299468 non-null  str  
 16  dateCreated  

In [84]:
df.isna().sum()

dateCrawled                0
name                       0
seller                     0
offerType                  0
price                      0
abtest                     0
vehicleType            37869
yearOfRegistration         0
gearbox                20209
powerPS                    0
model                  20484
kilometer                  0
monthOfRegistration        0
fuelType               33386
brand                      0
notRepairedDamage      72060
dateCreated                0
nrOfPictures               0
postalCode                 0
lastSeen                   0
dtype: int64

## Inital Observations
At first glance, we can see that some of the data is, unsurprisingly, in German. We may need to convert it to English later for legibility. Although there are null values present, it does not exceed 20% in any given column so there is still enough data to be usable. The column names are in camelcase, so we will change it to the standard Python snakecase to be more visually consistent.

In [85]:
df.columns

Index(['dateCrawled', 'name', 'seller', 'offerType', 'price', 'abtest',
       'vehicleType', 'yearOfRegistration', 'gearbox', 'powerPS', 'model',
       'kilometer', 'monthOfRegistration', 'fuelType', 'brand',
       'notRepairedDamage', 'dateCreated', 'nrOfPictures', 'postalCode',
       'lastSeen'],
      dtype='str')

In [86]:
new_index = ['date_crawled', 'name', 'seller', 'offer_type', 'price', 'ab_test',
       'vehicle_type', 'registration_year', 'gearbox', 'power_PS', 'model',
       'kilometer', 'registration_month', 'fuel_type', 'brand',
       'unrepaired_damage', 'ad_created', 'nr_of_pictures', 'postal_code',
       'last_seen']
df.columns = new_index
df.head()

,date_crawled,name,seller,offer_type,price,ab_test,vehicle_type,registration_year,gearbox,power_PS,model,kilometer,registration_month,fuel_type,brand,unrepaired_damage,ad_created,nr_of_pictures,postal_code,last_seen
0,2016-03-24 11:52:17,Golf_3_1.6,privat,Angebot,480,test,NaN,1993,manuell,0,golf,150000,0,benzin,volkswagen,NaN,2016-03-24 00:00:00,0,70435,2016-04-07 03:16:57
1,2016-03-24 10:58:45,A5_Sportback_2.7_Tdi,privat,Angebot,18300,test,coupe,2011,manuell,190,NaN,125000,5,diesel,audi,ja,2016-03-24 00:00:00,0,66954,2016-04-07 01:46:50
2,2016-03-14 12:52:21,"Jeep_Grand_Cherokee_""Overland""",privat,Angebot,9800,test,suv,2004,automatik,163,grand,125000,8,diesel,jeep,NaN,2016-03-14 00:00:00,0,90480,2016-04-05 12:47:46
3,2016-03-17 16:54:04,GOLF_4_1_4__3TÜRER,privat,Angebot,1500,test,kleinwagen,2001,manuell,75,golf,150000,6,benzin,volkswagen,nein,2016-03-17 00:00:00,0,91074,2016-03-17 17:40:17
4,2016-03-31 17:25:20,Skoda_Fabia_1.4_TDI_PD_Classic,privat,Angebot,3600,test,kleinwagen,2008,manuell,69,fabia,90000,7,diesel,skoda,nein,2016-03-31 00:00:00,0,60437,2016-04-06 10:17:21


In [87]:
df.describe(include='all')

,date_crawled,name,seller,offer_type,price,ab_test,vehicle_type,registration_year,gearbox,power_PS,model,kilometer,registration_month,fuel_type,brand,unrepaired_damage,ad_created,nr_of_pictures,postal_code,last_seen
count,371528,371528,371528,371528,3.715280e+05,371528,333659,371528.000000,351319,371528.000000,351044,371528.000000,371528.000000,338142,371528,299468,371528,371528.0,371528.00000,371528
unique,280500,233531,2,2,NaN,2,8,NaN,2,NaN,251,NaN,NaN,7,40,2,114,NaN,NaN,182806
top,2016-03-24 14:49:47,Ford_Fiesta,privat,Angebot,NaN,test,limousine,NaN,manuell,NaN,golf,NaN,NaN,benzin,volkswagen,nein,2016-04-03 00:00:00,NaN,NaN,2016-04-07 06:45:59
freq,7,657,371525,371516,NaN,192585,95894,NaN,274214,NaN,30070,NaN,NaN,223857,79640,263182,14450,NaN,NaN,17
mean,NaN,NaN,NaN,NaN,1.729514e+04,NaN,NaN,2004.577997,NaN,115.549477,NaN,125618.688228,5.734445,NaN,NaN,NaN,NaN,0.0,50820.66764,NaN
std,NaN,NaN,NaN,NaN,3.587954e+06,NaN,NaN,92.866598,NaN,192.139578,NaN,40112.337051,3.712412,NaN,NaN,NaN,NaN,0.0,25799.08247,NaN
min,NaN,NaN,NaN,NaN,0.000000e+00,NaN,NaN,1000.000000,NaN,0.000000,NaN,5000.000000,0.000000,NaN,NaN,NaN,NaN,0.0,1067.00000,NaN
25%,NaN,NaN,NaN,NaN,1.150000e+03,NaN,NaN,1999.000000,NaN,70.000000,NaN,125000.000000,3.000000,NaN,NaN,NaN,NaN,0.0,30459.00000,NaN
50%,NaN,NaN,NaN,NaN,2.950000e+03,NaN,NaN,2003.000000,NaN,105.000000,NaN,150000.000000,6.000000,NaN,NaN,NaN,NaN,0.0,49610.00000,NaN
75%,NaN,NaN,NaN,NaN,7.200000e+03,NaN,NaN,2008.000000,NaN,150.000000,NaN,150000.000000,9.000000,NaN,NaN,NaN,NaN,0.0,71546.00000,NaN


From the summary statistics we can see the nr_of_pictures column is all zeroes so we can remove it. The price and offer type columns can also be removed since almost all the values are the same.

In [88]:
df = df.drop(['nr_of_pictures', 'seller', 'offer_type'], axis=1)

## Exploring Columns
### Dates

In [90]:
df[['date_crawled', 'ad_created', 'last_seen']].head()

,date_crawled,ad_created,last_seen
0,2016-03-24 11:52:17,2016-03-24 00:00:00,2016-04-07 03:16:57
1,2016-03-24 10:58:45,2016-03-24 00:00:00,2016-04-07 01:46:50
2,2016-03-14 12:52:21,2016-03-14 00:00:00,2016-04-05 12:47:46
3,2016-03-17 16:54:04,2016-03-17 00:00:00,2016-03-17 17:40:17
4,2016-03-31 17:25:20,2016-03-31 00:00:00,2016-04-06 10:17:21


In [91]:
df['date_crawled'].str[:10].value_counts(normalize=True).sort_index()

date_crawled
2016-03-05    0.025551
2016-03-06    0.014462
2016-03-07    0.035690
2016-03-08    0.033454
2016-03-09    0.034210
2016-03-10    0.032563
2016-03-11    0.032727
2016-03-12    0.036194
2016-03-13    0.015735
2016-03-14    0.036275
2016-03-15    0.033451
2016-03-16    0.030148
2016-03-17    0.031661
2016-03-18    0.013127
2016-03-19    0.035292
2016-03-20    0.036353
2016-03-21    0.035739
2016-03-22    0.032469
2016-03-23    0.031971
2016-03-24    0.029912
2016-03-25    0.032934
2016-03-26    0.031965
2016-03-27    0.030275
2016-03-28    0.035112
2016-03-29    0.034170
2016-03-30    0.033529
2016-03-31    0.031885
2016-04-01    0.034116
2016-04-02    0.035079
2016-04-03    0.038735
2016-04-04    0.037612
2016-04-05    0.012823
2016-04-06    0.003163
2016-04-07    0.001620
Name: proportion, dtype: float64

The site was crawled every day for a month from March to April 2016 and the distribution of listings per day are fairly uniform.

In [92]:
df['last_seen'].str[:10].value_counts(normalize=True).sort_index()

last_seen
2016-03-05    0.001292
2016-03-06    0.004134
2016-03-07    0.005262
2016-03-08    0.008056
2016-03-09    0.009994
2016-03-10    0.011563
2016-03-11    0.013046
2016-03-12    0.023401
2016-03-13    0.008489
2016-03-14    0.012301
2016-03-15    0.016411
2016-03-16    0.016419
2016-03-17    0.028760
2016-03-18    0.006931
2016-03-19    0.016314
2016-03-20    0.019915
2016-03-21    0.020136
2016-03-22    0.020607
2016-03-23    0.018149
2016-03-24    0.019237
2016-03-25    0.019097
2016-03-26    0.016160
2016-03-27    0.016909
2016-03-28    0.022273
2016-03-29    0.023312
2016-03-30    0.023856
2016-03-31    0.024238
2016-04-01    0.024022
2016-04-02    0.025016
2016-04-03    0.025366
2016-04-04    0.025654
2016-04-05    0.126206
2016-04-06    0.217830
2016-04-07    0.129648
Name: proportion, dtype: float64

There are a disproportinate amount of last_seen values on the last 3 days, which is likely due to the crawling period ending rather than actual sales. This is further supported with the low number of date_crawled values on those days.

In [93]:
print(df['ad_created'].str[:10].value_counts(normalize=True).sort_index())
print(df['ad_created'].str[:10].value_counts(normalize=True).sort_values())

ad_created
2014-03-10    0.000003
2015-03-20    0.000003
2015-06-11    0.000003
2015-06-18    0.000003
2015-08-07    0.000003
                ...   
2016-04-03    0.038893
2016-04-04    0.037741
2016-04-05    0.011655
2016-04-06    0.003155
2016-04-07    0.001558
Name: proportion, Length: 114, dtype: float64
ad_created
2015-09-09    0.000003
2015-06-18    0.000003
2015-11-12    0.000003
2015-12-27    0.000003
2015-08-07    0.000003
                ...   
2016-03-21    0.035812
2016-03-12    0.036011
2016-03-20    0.036463
2016-04-04    0.037741
2016-04-03    0.038893
Name: proportion, Length: 114, dtype: float64


There is a wide range of ad_created dates, most fall within a 1-2 months of the listing date but some span back a year or more.

### Registration

In [94]:
df['registration_year'].describe()

count    371528.000000
mean       2004.577997
std          92.866598
min        1000.000000
25%        1999.000000
50%        2003.000000
75%        2008.000000
max        9999.000000
Name: registration_year, dtype: float64

The registration_year data has some odd values, the min is 1000 and the max is 9999 which is clearly incorrect. Since the most recent listings are from 2016, we can remove any values above 2016. Determining the earliest valid date is a bit harder. Realistically, vintage automobiles could be from around the early 1900s.

In [95]:
(~df['registration_year'].between(1900, 2016)).sum() / df.shape[0]

np.float64(0.03969552765874981)

Since less than 4% of the data is from before 1900 or after 2016, we can remove these rows.

In [96]:
df = df[df['registration_year'].between(1900, 2016)]
df['registration_year'].value_counts(normalize=True).head(10)

registration_year
2000    0.068813
1999    0.063812
2005    0.062548
2006    0.056702
2001    0.056668
2003    0.055701
2004    0.055345
2002    0.053784
1998    0.050314
2007    0.049535
Name: proportion, dtype: float64

Most vehicles were first registered within the last 20 years.

### Brand

In [97]:
df['brand'].value_counts(normalize=True)

brand
volkswagen        0.212391
bmw               0.109695
opel              0.107069
mercedes_benz     0.095955
audi              0.089352
ford              0.068849
renault           0.047559
peugeot           0.029859
fiat              0.025758
seat              0.018631
skoda             0.015407
mazda             0.015348
smart             0.014104
citroen           0.013874
nissan            0.013569
toyota            0.012747
sonstige_autos    0.010648
hyundai           0.009830
mini              0.009210
volvo             0.009129
mitsubishi        0.008266
honda             0.007587
kia               0.006878
alfa_romeo        0.006348
suzuki            0.006318
porsche           0.006149
chevrolet         0.005014
chrysler          0.003941
dacia             0.002450
jeep              0.002186
daihatsu          0.002186
land_rover        0.002136
subaru            0.002125
jaguar            0.001721
trabant           0.001620
saab              0.001452
daewoo            0.00

German manufacturers dominate almost half of the listings, representing 4 out of the top 5 brands. Volkswagen is the most popular brand, with approximately double the cars for sale of the next top 2 brands combined. Lots of brands have a very low percentage of listings, so we'll limit our analysis to brands that have more than 5% of total listings.

In [98]:
brand_counts = df['brand'].value_counts(normalize=True)
common_brands = brand_counts[brand_counts > 0.05].index
print(common_brands)

Index(['volkswagen', 'bmw', 'opel', 'mercedes_benz', 'audi', 'ford'], dtype='str', name='brand')


In [102]:
brand_mean_prices = {}
for brand in common_brands:
    brand_only = df[df['brand'] == brand]
    mean_price = brand_only['price'].mean()
    brand_mean_prices[brand] = int(mean_price)

brand_mean_prices

{'volkswagen': 13643,
 'bmw': 14798,
 'opel': 3248,
 'mercedes_benz': 17614,
 'audi': 16218,
 'ford': 8702}

There's a distinct price gap apparent between the brands. Ford and Opel are more budget friendly whereas BMW, Mercedes Benz, and Audi are more on the luxury end. Volkswagen sits in the middle of the price range, which may account for its popularity.

### Mileage

In [103]:
print(df['kilometer'].value_counts())
print(df['kilometer'].describe())

kilometer
150000    230178
125000     36652
100000     15449
90000      12149
80000      10756
70000       9534
60000       8505
50000       7441
5000        6593
40000       6269
30000       5901
20000       5506
10000       1847
Name: count, dtype: int64
count    356780.000000
mean     125416.405068
std       40158.087893
min        5000.000000
25%      100000.000000
50%      150000.000000
75%      150000.000000
max      150000.000000
Name: kilometer, dtype: float64


The values in the kilometer column appear to be rounded, and most of the cars for sale have a higher mileage (or would it be kilometerage...? I am ignorantly American). The column appears to have clean data with no null or 0 values.

In [118]:
brand_mean_mileage = {}

for brand in common_brands:
    brand_only = df[df['brand'] == brand]
    mean_mileage = brand_only['kilometer'].mean()
    brand_mean_mileage[brand] = int(mean_mileage)

mean_mileage = pd.Series(brand_mean_mileage).sort_values(ascending=False)
mean_prices = pd.Series(brand_mean_prices).sort_values(ascending=False)

In [120]:
brand_info = pd.DataFrame({
    'mean_mileage': mean_mileage,
    'mean_prices': mean_prices
})
brand_info.sort_values('mean_mileage')

,mean_mileage,mean_prices
ford,123618,8702
volkswagen,128337,13643
opel,128755,3248
audi,129491,16218
mercedes_benz,130580,17614
bmw,132657,14798


The mileage across brands does not vary as much as pricing. However, there is a slight trend where more expensive vehicles have higher mileage and less expensive vehicles have lower mileage.